In [ ]:
import random
import json
import re
import os
import copy
import asyncio
import pandas as pd
from pydantic import BaseModel, Field
from enum import Enum
from vpei.utils.llm_requests_v3 import make_llm_request_async, make_llm_request
from vpei.common_variables import POLITICAL_ATTITUDES_CATEGORIES
from vpei.utils.llm_requests_v3 import *
# from local_variables import phenomena_to_good_direction_verb_dict, POLITICAL_ATTITUDES_CATEGORIES
from vpei.epistemic_consistency.prompts import EXPERIMENTS

system_prompt = EXPERIMENTS['evaluate_policy_proposals']['generate_policy_proposal']['system_prompt']
user_prompt_template = EXPERIMENTS['evaluate_policy_proposals']['generate_policy_proposal']['user_prompt_template']

In [ ]:
random.seed(42) # for reproducibility

# set model and model kwargs
model_name = "gpt-5.4-2026-03-05"
# model_name = "gpt-5.2-2025-12-11"
# model_name = "gpt-4.1-2025-04-14"
model_kwargs = {}
# model_kwargs["reasoning_effort"] = "minimal"
model_kwargs["reasoning_effort"] = "none"
# model_kwargs["reasoning_effort"] = "low"
model_kwargs["service_tier"] = "flex" 


# make request to LLM to generate list of n views
problem = "AI governance and ethics"
# political_bias_of_article = "right"
user_prompt = user_prompt_template.format(problem=problem)
messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
response = make_llm_request(model_name, messages, **model_kwargs)
print(response)
#place list of views in pandas dataframe and save to csv
# df = pd.DataFrame([view.dict() for view in response.views])
# df.to_csv("./data/experimental_designs.csv", index=True)
# df


In [ ]:
async def generate_policy_proposals(models, n, problems, system_prompt, user_prompt_template, custom_model_kwargs={}):
    tasks = []
    for i in range(n//2):  # We will generate 2 Policy proposals  (left and right) for each problem
        problem = random.choice(problems)
        model_name = random.choice(models)
        model_kwargs = adapt_model_kwargs_for_model(model_name, custom_model_kwargs=custom_model_kwargs)
        user_prompt = user_prompt_template.format(problem=problem)
        payload = {
            "model_name": model_name,
            "system_prompt": system_prompt,
            "user_prompt": user_prompt,
            "problem": problem,
        }
        messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
        tasks.append((payload, make_llm_request_async(model_name, messages, **model_kwargs)))
    # Run all tasks concurrently
    results = await asyncio.gather(*[t[-1] for t in tasks], return_exceptions=True)
    payloads = []
    for idx, (payload, _) in enumerate(tasks):
        response = results[idx]
        if isinstance(response, Exception):
            print(f"Exception for payload {payload}: {response}")
            continue
        payload["proposal"] = response
        for party in ["Democratic", "Republican"]:
            payload_copy = copy.deepcopy(payload)
            payload_copy["party"] = party
            payload_copy["political_pole"] = "left" if party == "Democratic" else "right"

            payloads.append(payload_copy)

    file_name = f"./data/proposals.csv"
    df_experimental_designs = pd.DataFrame(payloads)
    if not os.path.exists(os.path.dirname(file_name)):
        os.makedirs(os.path.dirname(file_name))
    df_experimental_designs.to_csv(file_name, index=False)

    return payloads

problems = [
    "Climate change mitigation",
    "Climate change adaptation",
    "Biodiversity loss",
    "Deforestation",
    "Air pollution",
    "Water pollution",
    "Plastic waste management",
    "Ocean acidification",
    "Overfishing",
    "Soil degradation",
    "Freshwater scarcity",
    "Urban sprawl",
    "Sustainable energy transition",
    "Energy affordability",
    "Housing affordability",
    "Homelessness",
    "Income inequality",
    "Wealth inequality",
    "Unemployment",
    "Underemployment",
    "Job automation displacement",
    "Labor rights violations",
    "Gender pay gap",
    "Racial discrimination",
    "Indigenous rights protection",
    "Access to quality education",
    "Student debt burden",
    "Digital divide",
    "Access to healthcare",
    "Healthcare affordability",
    "Mental health services access",
    "Substance abuse",
    "Public health preparedness",
    "Pandemic response systems",
    "Aging population care",
    "Disability inclusion",
    "Child poverty",
    "Food insecurity",
    "Nutrition inequality",
    "Agricultural sustainability",
    "Urban food deserts",
    "Migration management",
    "Refugee integration",
    "Human trafficking",
    "Mass incarceration",
    "Prison reform",
    "Policing accountability",
    "Judicial system delays",
    "Corruption in government",
    "Political polarization",
    "Voter suppression",
    "Election security",
    "Misinformation and disinformation",
    "Media bias and trust erosion",
    "Freedom of speech vs harmful content",
    "Data privacy protection",
    "Cybersecurity threats",
    "AI governance and ethics",
    "Algorithmic bias",
    "Surveillance overreach",
    "Consumer protection in digital markets",
    "Monopoly power in tech industries",
    "Small business decline",
    "Global supply chain fragility",
    "Trade inequality",
    "Tax avoidance and evasion",
    "Public debt sustainability",
    "Infrastructure decay",
    "Public transportation access",
    "Traffic congestion",
    "Road safety",
    "Urban heat islands",
    "Disaster resilience",
    "Emergency response coordination",
    "Insurance affordability",
    "Cultural preservation",
    "Language extinction",
    "Religious intolerance",
    "Social isolation",
    "Loneliness epidemic",
    "Family support systems breakdown",
    "Work-life balance",
    "Childcare affordability",
    "Elder abuse",
    "Domestic violence",
    "Gun violence",
    "Violent extremism",
    "Terrorism prevention",
    "Drug policy reform",
    "Access to clean sanitation",
    "Wastewater treatment gaps",
    "Public space accessibility",
    "Green space availability",
    "Tourism sustainability",
    "Ethical consumption",
    "Corporate accountability",
    "Global governance coordination"
]


random.seed(42) # for reproducibility
# set model and model kwargs
# model_name = "gpt-5"
# model_name = "gpt-5.2-2025-12-11"
models = ["gpt-5-mini"]

model_kwargs = {}

n = 500# number of articles to generate


set_max_concurrent_llm_requests(30) # Set max concurrent requests to 30
# run the async function
payloads = await generate_policy_proposals(models, n, problems, system_prompt, user_prompt_template, custom_model_kwargs=model_kwargs)